# Il repository come memoria: istruzioni e fonte di verità

Il codice del capitolo [«Il repository come memoria: istruzioni e fonte di verità»](https://book.paithon.it/main/Harness/il-repository-come-memoria.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Il repository come memoria: istruzioni e fonte di verità

[Leggi la pagina](https://book.paithon.it/main/Harness/il-repository-come-memoria.html)


### Tenere vera la memoria


In [ ]:
import re

# Un repository finto: percorso -> contenuto. L'indice cita i documenti e i
# comandi fra apici inversi, come si fa di solito in Markdown.
repo = {
    "AGENTS.md": (
        "# Negozio online\n"
        "Avvio: `make dev`. Controllo: `make check`, poi `make lint`.\n"
        "Mai scrivere sul database di produzione.\n"
        "- dove va il codice nuovo: `docs/architettura.md`\n"
        "- prima di toccare il checkout: `docs/pagamenti.md`\n"
        "- a inizio sessione: `docs/piani/carrello.md`\n"
    ),
    "docs/architettura.md": "Tre strati: dati, servizi, interfaccia.",
    "docs/piani/carrello.md": "Fatto: aggiunta al carrello. Manca: sconti.",
    "docs/sicurezza.md": "Le chiavi dei pagamenti non vanno mai nei file.",
    "Makefile": "dev:\n\tpython app.py\ncheck:\n\tpytest\n",
}

indice = repo["AGENTS.md"]
citati = set(re.findall(r"`(docs/[^`]+)`", indice))
documenti = {p for p in repo if p.startswith("docs/")}
comandi = set(re.findall(r"`make (\w+)`", indice))
bersagli = set(re.findall(r"^(\w+):", repo["Makefile"], re.M))

problemi = {
    "documenti citati ma assenti": sorted(citati - set(repo)),
    "documenti mai citati": sorted(documenti - citati),
    "comandi citati ma assenti": sorted(comandi - bersagli),
}
print("righe dell'indice:", len(indice.splitlines()))
for nome, elenco in problemi.items():
    print(f"{nome}: {elenco}")
print("cancello:", "chiuso" if any(problemi.values()) else "aperto")

## Il primo turno: preparare prima di costruire

[Leggi la pagina](https://book.paithon.it/main/Harness/il-primo-turno.html)


### Un cancello per l'elenco


In [ ]:
import copy

# L'elenco com'era nell'ultimo salvataggio: il ciclo lo rilegge dalla storia
# di git, non dal file che la sessione ha avuto in mano.
elenco = [
    {"category": "funzionale",
     "description": "Un utente aggiunge un prodotto e vede il totale",
     "steps": ["Apri un prodotto", "Premi «Aggiungi»", "Controlla il totale"],
     "passes": False},
    {"category": "funzionale",
     "description": "Un utente paga con la carta",
     "steps": ["Apri il carrello", "Inserisci la carta", "Ricevi la conferma"],
     "passes": False},
    {"category": "interfaccia",
     "description": "La pagina del carrello si legge su un telefono",
     "steps": ["Apri il carrello a 390 pixel", "Nessun testo esce dallo schermo"],
     "passes": False},
]

def controlla(prima, dopo):
    """Rifiuta ogni cambiamento che non sia il valore vero/falso di `passes`."""
    errori = []
    nomi_prima = [v["description"] for v in prima]
    nomi_dopo = [v["description"] for v in dopo]
    if len(set(nomi_dopo)) != len(nomi_dopo):
        errori.append("due voci con la stessa descrizione")
    for nome in sorted(set(nomi_prima) - set(nomi_dopo)):
        errori.append(f"voce tolta: {nome}")
    for nome in sorted(set(nomi_dopo) - set(nomi_prima)):
        errori.append(f"voce nuova o descrizione riscritta: {nome}")
    comuni = set(nomi_prima) & set(nomi_dopo)
    ordine_prima = [n for n in nomi_prima if n in comuni]
    ordine_dopo = [n for n in nomi_dopo if n in comuni]
    if ordine_prima != ordine_dopo:
        errori.append("voci riordinate")
    vecchie = {v["description"]: v for v in prima}
    nuove = {v["description"]: v for v in dopo}
    for nome in sorted(comuni):
        a, b = vecchie[nome], nuove[nome]
        if a.keys() != b.keys():
            errori.append(f"campi aggiunti o tolti in: {nome}")
            continue
        for campo in sorted(a):
            if campo != "passes" and a[campo] != b[campo]:
                errori.append(f"campo «{campo}» cambiato in: {nome}")
        if not isinstance(b["passes"], bool):
            errori.append(f"`passes` non è vero o falso in: {nome}")
    return errori

def avanzamento(voci):
    """Il contatore ingenuo: conta sull'elenco com'è adesso."""
    return f"{sum(v['passes'] is True for v in voci)}/{len(voci)}"

# Sessione onesta: il carrello ora funziona, e si spunta soltanto quello.
onesta = copy.deepcopy(elenco)
onesta[0]["passes"] = True

# Sessione furba: il pagamento è difficile e sparisce; la prova del carrello
# perde il controllo del totale, e il carrello risulta fatto.
furba = copy.deepcopy(elenco)
del furba[1]
furba[0]["steps"] = ["Apri un prodotto", "Premi «Aggiungi»"]
furba[0]["passes"] = True

for nome, dopo in [("onesta", onesta), ("furba", furba)]:
    errori = controlla(elenco, dopo)
    esito = "accettata" if not errori else "RIFIUTATA"
    print(f"sessione {nome}: avanzamento {avanzamento(dopo)}, {esito}")
    for e in errori:
        print("   ", e)

## Lavorare a turni: il passaggio di consegne

[Leggi la pagina](https://book.paithon.it/main/Harness/passaggio-di-consegne.html)


### Compattare o ripartire da capo


In [ ]:
import math

# Un modello a numeri tondi, non una misura: a ogni giro il modello rilegge
# tutto il contesto, e la cronologia cresce della sua risposta e dell'esito
# di uno strumento.
FINESTRA = 100_000      # token che il modello legge al massimo in una volta
FISSO = 5_000           # istruzioni e catalogo degli strumenti, a ogni giro
GIRO = 3_000            # quanto cresce la cronologia a ogni giro
GIRI = 60               # la lunghezza del compito
RIASSUNTO = 4_000       # la cronologia compattata diventa un riassunto
REGISTRO = 2_000        # registro, voci aperte dell'elenco, storia recente
PER_SESSIONE = 15       # giri di una sessione: una funzionalità

def senza_niente():
    letti, storia, contesti = 0, 0, []
    for t in range(GIRI):
        contesto = FISSO + storia
        if contesto > FINESTRA:          # non ci sta più: il lavoro si ferma
            break
        contesti.append(contesto)
        letti += contesto
        storia += GIRO
    return letti, contesti

def con_compattazione(soglia):
    letti, storia, contesti, volte, per_riassunti = 0, 0, [], 0, 0
    for t in range(GIRI):
        if FISSO + storia > soglia * FINESTRA:
            per_riassunti += FISSO + storia   # la chiamata che lo scrive
            storia, volte = RIASSUNTO, volte + 1
        contesto = FISSO + storia
        contesti.append(contesto)
        letti += contesto
        storia += GIRO
    return letti + per_riassunti, contesti, volte, per_riassunti

def con_ripartenza(giri=GIRI, per_sessione=PER_SESSIONE):
    letti, contesti = 0, []
    for t in range(giri):
        dentro = t % per_sessione        # giri già fatti in questa sessione
        contesto = FISSO + REGISTRO + dentro * GIRO
        contesti.append(contesto)
        letti += contesto
    return letti, contesti, math.ceil(giri / per_sessione)

def it(n):                               # 1648000 -> 1.648.000
    return f"{n:,}".replace(",", ".")

a, ca = senza_niente()
b, cb, volte, _ = con_compattazione(0.8)
b5, cb5, volte5, scrivere5 = con_compattazione(0.5)
c, cc, sessioni = con_ripartenza()
meta = len(ca) // 2
print(f"senza niente: si ferma dopo {len(ca)} giri di {GIRI}, "
      f"letti {it(a)} ({it(sum(ca[:meta]))} nei primi {meta})")
print(f"compattazione all'80%: {volte} riassunti, picco {it(max(cb))}, "
      f"letti {it(b)}")
print(f"compattazione al 50%: {volte5} riassunti, picco {it(max(cb5))}, "
      f"letti {it(b5)} ({it(scrivere5)} per scriverli)")
print(f"ripartenza: {sessioni} sessioni, picco {it(max(cc))}, letti {it(c)}")
print(f"letti per giro, in media: {it(b // GIRI)}, {it(b5 // GIRI)}, "
      f"{it(c // GIRI)}")

# Due voci nella stessa sessione, contro due sessioni da una voce.
insieme = con_ripartenza(2 * PER_SESSIONE, 2 * PER_SESSIONE)[0]
separate = con_ripartenza(2 * PER_SESSIONE, PER_SESSIONE)[0]
print(f"due voci: {it(insieme)} in una sessione, {it(separate)} in due")

## Finito vuol dire collaudato: la verifica di un lavoro lungo

[Leggi la pagina](https://book.paithon.it/main/Harness/finito-vuol-dire-collaudato.html)


### Il collaudo da utente


In [ ]:
# Due componenti del negozio, scritti in due sessioni diverse.
class Carrello:
    """Tiene i prezzi in centesimi, come si fa per non perdere spiccioli."""
    def __init__(self):
        self.righe = []
    def aggiungi(self, nome, prezzo_euro):
        self.righe.append((nome, round(prezzo_euro * 100)))
    def totale(self):
        return sum(centesimi for _, centesimi in self.righe)

def pagina_totale(carrello):
    """La pagina del carrello: chi l'ha scritta credeva il totale in euro."""
    return f"Totale: {carrello.totale():.2f} €".replace(".", ",")

# I test d'unità: ciascuno prova il suo pezzo da solo.
def test_carrello():
    c = Carrello()
    c.aggiungi("tazza", 19.99)
    return c.totale() == 1999                     # centesimi: giusto

class CarrelloFinto:                              # la controfigura del test
    def totale(self):                             # della pagina: risponde in
        return 19.99                              # euro, come la pagina crede

def test_pagina():
    return pagina_totale(CarrelloFinto()) == "Totale: 19,99 €"

# Il collaudo della voce: i pezzi veri, montati insieme, e i suoi passi di
# prova. Qui si legge lo schermo come una stringa; un harness vero passa da
# un browser.
def collaudo_carrello():
    c = Carrello()
    c.aggiungi("tazza", 19.99)                    # «aggiungi una tazza»
    schermo = pagina_totale(c)
    return schermo == "Totale: 19,99 €", schermo  # «controlla il totale»

voce = {"description": "Un utente aggiunge un prodotto e vede il totale",
        "steps": ["Aggiungi una tazza da 19,99 €",
                  "Controlla che il totale sia il prezzo del prodotto"],
        "passes": False}
collaudi = {voce["description"]: collaudo_carrello}   # un collaudo per voce

def chiudi_voce(voce, dichiarazione_agente):
    """Il campo `passes` lo scrive il collaudo della voce; la dichiarazione
    dell'agente non entra nel conto."""
    ok, schermo = collaudi[voce["description"]]()
    voce["passes"] = ok
    return ok, schermo

print("test d'unità del carrello:", "passa" if test_carrello() else "fallisce")
print("test d'unità della pagina:", "passa" if test_pagina() else "fallisce")
ok, schermo = chiudi_voce(voce, dichiarazione_agente="fatto, i test passano")
print("collaudo da utente:", "passa" if ok else "fallisce",
      f"(lo schermo dice «{schermo}»)")
print("passes:", voce["passes"])

## Quando nessuno guarda: osservabilità ed entropia

[Leggi la pagina](https://book.paithon.it/main/Harness/quando-nessuno-guarda.html)


### Seguire una richiesta da capo a fondo


In [ ]:
import itertools
import json

contatore = itertools.count(1)
eventi = []                                   # il giornale degli eventi

def registra(richiesta, fase, **campi):
    """Un evento per fase, sempre con l'identificativo della richiesta."""
    eventi.append({"richiesta": richiesta, "fase": fase, **campi})

# L'archivio dei documenti, codificati mesi fa in vettori da 8 numeri.
archivio = {f"doc{i}": [1.0] * 8 for i in range(1000)}

def codifica(testo):
    """Il codificatore, aggiornato ieri: ora produce vettori da 4."""
    return [0.5] * 4

def cerca(richiesta, domanda):
    q = codifica(domanda)
    registra(richiesta, "codifica", lunghezza=len(q))
    candidati, scartate = [], set()           # lunghezze dei vettori scartati
    for nome, v in archivio.items():
        if len(v) != len(q):                  # «prudente»: salta in silenzio
            scartate.add(len(v))
            continue
        candidati.append(nome)
    registra(richiesta, "recupero", visti=len(archivio),
             compatibili=len(candidati), scartate=sorted(scartate))
    return candidati

def rispondi(domanda):
    richiesta = f"r{next(contatore)}"         # lega fra loro gli eventi
    documenti = cerca(richiesta, domanda)
    registra(richiesta, "risposta", documenti=len(documenti))
    if not documenti:
        return "Non ho trovato documenti pertinenti."
    return f"Ecco {len(documenti)} documenti."

# Quello che vede chi usa il servizio: una risposta plausibile.
print(rispondi("Come si restituisce un prodotto difettoso?"))

# Quello che dice il giornale, seguendo la richiesta da capo a fondo.
traccia = [e for e in eventi if e["richiesta"] == "r1"]
for e in traccia:
    print(json.dumps(e, ensure_ascii=False))

dove = next(e for e in traccia if e.get("compatibili") == 0)
print(f"si azzera in «{dove['fase']}»: domanda da {traccia[0]['lunghezza']} "
      f"numeri, archivio da {dove['scartate'][0]}")

### L'entropia, e la pulizia che non finisce


In [ ]:
# Un modello a numeri tondi, non una misura: b è la frazione del codice che
# segue uno schema storto. A ogni sessione si riscrive una parte del codice,
# e il codice nuovo copia gli schemi che trova.
RISCRITTO = 0.2         # frazione del codice riscritta a ogni sessione
ERRORE = 0.1            # probabilità di sbagliare copiando uno schema sano
CONTAGIO = 0.4          # quanto più volentieri si copia una scorciatoia
ONDA = 5                # la pulizia a ondate passa ogni cinque sessioni
SESSIONI, REGIME = 600, 300          # le medie si prendono sulle ultime 300

def copia(b, contagio):
    """Probabilità di copiare uno schema storto, se lo è la frazione b."""
    return b + contagio * b * (1 - b)

def deriva(contagio, permanenza=None, a_ondate=False, sessioni=SESSIONI):
    """La frazione storta sessione per sessione. La pulizia corregge ogni
    volta una quota del codice storto, scelta perché uno schema storto resti
    in media `permanenza` sessioni prima di essere corretto."""
    if permanenza is None:
        quota = 0.0
    elif a_ondate:      # in media si aspetta (ONDA - 1) / 2 la prima ondata
        quota = ONDA / (permanenza + (ONDA - 1) / 2)
    else:
        quota = 1 / permanenza
    b, storia = 0.0, []
    for k in range(1, sessioni + 1):
        c = copia(b, contagio)
        nuovo = c + ERRORE * (1 - c)     # il codice riscritto è storto
        tolta = quota if (k % ONDA == 0 or not a_ondate) else 0.0
        b = (1 - RISCRITTO) * b + RISCRITTO * nuovo - tolta * b
        storia.append(b)
    return storia

def media(storia):
    return sum(storia[-REGIME:]) / REGIME

def it(x):                               # 0.167 -> 0,167
    return f"{x:.3f}".replace(".", ",")

print("permanenza   in proporzione   con contagio")
for p in (2, 5, 10, 20):
    print(f"{p:>2} sessioni {it(media(deriva(0, p))):>16} "
          f"{it(media(deriva(CONTAGIO, p))):>14}")
for contagio, nome in [(0, "in proporzione"), (CONTAGIO, "con contagio")]:
    onde = deriva(contagio, 10, a_ondate=True)[-REGIME:]
    print(f"a ondate, permanenza 10, {nome}: media {it(media(onde))}, "
          f"picco {it(max(onde))}")
print(f"senza pulizia, dopo {SESSIONI} sessioni: "
      f"{it(deriva(0)[-1])} e {it(deriva(CONTAGIO)[-1])}")

## Dal ciclo al grafo: il disegno di un lavoro che dura

[Leggi la pagina](https://book.paithon.it/main/Harness/dal-ciclo-al-grafo.html)


### Il grafo come struttura dati


In [ ]:
# Un grafo minimo per un lavoro lungo. I nodi leggono lo stato e restituiscono
# un aggiornamento; ogni campo dello stato ha la sua regola di fusione.
FUSIONE = {
    "registro": lambda vecchio, nuovo: vecchio + nuovo,    # si accoda
    "tentativi": lambda vecchio, nuovo: vecchio + nuovo,   # si somma
    "esito": lambda vecchio, nuovo: nuovo,                 # si sostituisce
}

def fondi(stato, aggiornamento):
    nuovo = dict(stato)
    for campo, valore in aggiornamento.items():
        nuovo[campo] = FUSIONE[campo](stato[campo], valore)
    return nuovo

def pianifica(stato):
    return {"registro": ["piano: una voce, il totale del carrello"]}

def costruisci(stato):
    tentativo = stato["tentativi"] + 1
    return {"registro": [f"tentativo {tentativo} costruito"], "tentativi": 1}

guasti = {"collauda": 1}             # il primo collaudo si interrompe

def collauda(stato):
    if guasti["collauda"]:
        guasti["collauda"] -= 1
        raise RuntimeError("la macchina si è spenta")     # un guasto simulato
    tentativo = stato["tentativi"]
    esito = "passato" if tentativo >= 2 else "fallito"
    riga = f"collaudo del tentativo {tentativo}: {esito}"
    return {"esito": esito, "registro": [riga]}

def rilascia(stato):
    return {"registro": ["rilasciato ai clienti"]}

NODI = {"pianifica": pianifica, "costruisci": costruisci,
        "collauda": collauda, "rilascia": rilascia}
ARCHI = {"pianifica": "costruisci", "costruisci": "collauda", "rilascia": None}
TETTO = 3                            # tentativi al massimo, poi ci si ferma

def dopo_il_collaudo(stato):         # l'unico arco che dipende dallo stato
    if stato["esito"] == "passato":
        return "rilascia"
    return "costruisci" if stato["tentativi"] < TETTO else None

CONDIZIONALI = {"collauda": dopo_il_collaudo}
IRREVERSIBILI = {"rilascia"}         # prima di questi serve una persona

# I punti salvati, uno per filo di lavoro. Qui è un dizionario; in un harness
# vero è un file o un database, scritto prima di passare al nodo dopo.
archivio = {}

def esegui(filo, approvati=()):
    """Riprende il filo dall'ultimo punto salvato e va avanti finché può.
    `approvati` sono i nodi irreversibili che una persona ha approvato."""
    inizio = {"registro": [], "tentativi": 0, "esito": None}
    nodo, stato = archivio.get(filo, ("pianifica", inizio))
    fatti = []
    while nodo is not None:
        if nodo in IRREVERSIBILI:
            if nodo not in approvati:
                return fatti, f"fermo prima di «{nodo}»: serve un'approvazione"
            stato = fondi(stato, {"registro": [f"«{nodo}» approvato"]})
        try:
            aggiornamento = NODI[nodo](stato)
        except RuntimeError as guasto:
            return fatti, f"guasto in «{nodo}»: {guasto}"
        stato = fondi(stato, aggiornamento)
        esito = aggiornamento.get("esito")
        fatti.append(f"{nodo} ({esito})" if esito else nodo)
        if nodo in CONDIZIONALI:
            nodo = CONDIZIONALI[nodo](stato)
        else:
            nodo = ARCHI[nodo]
        archivio[filo] = (nodo, stato)        # salvato dopo ogni passo
    if stato["esito"] == "fallito":           # uscito dal ciclo per il tetto
        return fatti, f"fermo dopo {TETTO} tentativi: serve una persona"
    return fatti, "fine"

for giro, approvati in [("prima corsa", ()), ("ripresa", ()),
                        ("dopo l'approvazione", ("rilascia",))]:
    fatti, motivo = esegui("ordine-17", approvati)
    print(f"{giro}: {' -> '.join(fatti)}")
    print(f"   {motivo}")
    if giro == "prima corsa":
        nodo, stato = archivio["ordine-17"]
        print(f"   punto salvato: si riparte da «{nodo}», "
              f"tentativi costruiti: {stato['tentativi']}")
print("registro del filo:")
for riga in archivio["ordine-17"][1]["registro"]:
    print(f"   {riga}")